# 🌸 Iris Flower Classification Project

> A complete end-to-end machine learning project using the classic Iris dataset — covering EDA, feature correlation, KNN with hyperparameter tuning, and Random Forest classification.

---

**Dataset:** Iris (built-in from sklearn)  
**Algorithms:** K-Nearest Neighbors (KNN) + Random Forest  
**Goal:** Classify iris flowers into 3 species based on petal and sepal features

## 📦 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier

print('✅ All libraries imported successfully!')

## 📂 2. Load Dataset

The Iris dataset is loaded directly from `sklearn.datasets`. It contains **150 samples** with **4 features** across **3 species**.

In [ ]:
iris = load_iris()
df = pd.DataFrame(iris.data, columns=iris.feature_names)
df['species'] = iris.target

# Map species numbers to names for readability
species_map = {0: 'Setosa', 1: 'Versicolor', 2: 'Virginica'}
df['species_name'] = df['species'].map(species_map)

print('First 5 rows:')
df.head()

## 🔍 3. Dataset Overview

In [ ]:
print('Shape:', df.shape)
print('\nClass Distribution:')
print(df['species_name'].value_counts())
print('\nDataset Info:')
df.info()

## 📊 4. Statistical Summary

In [ ]:
df.describe()

## 🎨 5. Exploratory Data Analysis (EDA)

### 5.1 Pair Plot — Feature Relationships by Species

In [ ]:
sns.pairplot(df, hue='species_name', palette='Set2',
             vars=iris.feature_names)
plt.suptitle('Pair Plot of Iris Features by Species', y=1.02, fontsize=14)
plt.tight_layout()
plt.show()

### 5.2 Correlation Heatmap

In [ ]:
plt.figure(figsize=(7, 5))
sns.heatmap(df[iris.feature_names].corr(), annot=True,
            cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Feature Correlation Heatmap', fontsize=13)
plt.tight_layout()
plt.show()

### 5.3 Boxplots — Feature Distribution per Species

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, feature in zip(axes.flatten(), iris.feature_names):
    sns.boxplot(data=df, x='species_name', y=feature, palette='Set2', ax=ax)
    ax.set_title(feature)
plt.suptitle('Feature Distribution by Species', fontsize=14)
plt.tight_layout()
plt.show()

## ⚙️ 6. Data Preprocessing

- Split features (X) and target (y)
- Train/Test split: **80% / 20%**
- Feature scaling using `StandardScaler`

In [ ]:
X = df[iris.feature_names]
y = df['species']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

print(f'Training samples : {len(X_train)}')
print(f'Testing  samples : {len(X_test)}')

## 🤖 7. Model 1 — K-Nearest Neighbors (KNN)

Using `GridSearchCV` to find the optimal `k` value (1 to 20) with 5-fold cross-validation.

In [ ]:
knn = KNeighborsClassifier()
param_grid = {'n_neighbors': range(1, 21)}
grid = GridSearchCV(knn, param_grid, cv=5)
grid.fit(X_train, y_train)

best_knn = grid.best_estimator_
y_pred_knn = best_knn.predict(X_test)

print('===== KNN Results =====')
print('Best K :', grid.best_params_)
print('Accuracy:', round(accuracy_score(y_test, y_pred_knn) * 100, 2), '%')
print('\nClassification Report:')
print(classification_report(y_test, y_pred_knn,
      target_names=['Setosa','Versicolor','Virginica']))

### 7.1 KNN Confusion Matrix

In [ ]:
plt.figure(figsize=(5, 4))
sns.heatmap(confusion_matrix(y_test, y_pred_knn),
            annot=True, fmt='d', cmap='Blues',
            xticklabels=['Setosa','Versicolor','Virginica'],
            yticklabels=['Setosa','Versicolor','Virginica'])
plt.title('KNN Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()

## 🌲 8. Model 2 — Random Forest Classifier

In [ ]:
rf = RandomForestClassifier(random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

print('===== Random Forest Results =====')
print('Accuracy:', round(accuracy_score(y_test, y_pred_rf) * 100, 2), '%')
print('\nClassification Report:')
print(classification_report(y_test, y_pred_rf,
      target_names=['Setosa','Versicolor','Virginica']))

### 8.1 Feature Importance (Random Forest)

In [ ]:
importances = rf.feature_importances_
feat_df = pd.DataFrame({'Feature': iris.feature_names, 'Importance': importances})
feat_df = feat_df.sort_values('Importance', ascending=False)

plt.figure(figsize=(7, 4))
sns.barplot(data=feat_df, x='Importance', y='Feature', palette='viridis')
plt.title('Feature Importance — Random Forest')
plt.tight_layout()
plt.show()

## 📋 9. Model Comparison

In [ ]:
knn_acc = round(accuracy_score(y_test, y_pred_knn) * 100, 2)
rf_acc  = round(accuracy_score(y_test, y_pred_rf)  * 100, 2)

comparison = pd.DataFrame({
    'Model': ['KNN (Best K)', 'Random Forest'],
    'Accuracy (%)': [knn_acc, rf_acc]
})
print(comparison.to_string(index=False))

# Bar chart
plt.figure(figsize=(6, 4))
sns.barplot(data=comparison, x='Model', y='Accuracy (%)', palette='Set2')
plt.ylim(90, 102)
plt.title('Model Accuracy Comparison')
plt.tight_layout()
plt.show()

## ✅ 10. Conclusion

| Model | Accuracy |
|---|---|
| KNN (GridSearchCV) | ~96–100% |
| Random Forest | ~96–100% |

- Both models perform excellently on the Iris dataset.
- **Petal length** and **petal width** are the most important features.
- **Setosa** is perfectly separable; **Versicolor** and **Virginica** have slight overlap.
- Random Forest is more robust due to ensemble learning.

> 🌸 *Project completed successfully!*